# GeneWeaver: Kaggle GPU runner

**Settings → Accelerator: GPU T4 x2, Internet: On.**

Run top to bottom. After every push from the Mac, re-run from the *Get the code* cell (it does `git pull`).

## 1. Check the GPUs
Should list 2 × Tesla T4.

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.exists("GeneWeaver"):
    !git clone https://github.com/shravani-oss/GeneWeaver.git
%cd /kaggle/working/GeneWeaver
!git pull
!git log --oneline -3

## 3. Install

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt
!pip install -q -e .
!python --version
!python -c "from numba import cuda; print('CUDA available:', cuda.is_available()); cuda.detect()"

## 4. Download chr21 (~46 M bases)

In [ ]:
!mkdir -p data
![ -f data/chr21.fa ] || (wget -q -O data/chr21.fa.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr21.fa.gz && gunzip data/chr21.fa.gz)
!ls -lh data/

## 5. Run all tests on the real GPU
`test_gpu_matches_cpu.py` now runs for real (no simulator), including the chr21 GPU == CPU test.
`-rs` lists anything skipped. Nothing in `test_gpu_matches_cpu.py` should be skipped here.

In [ ]:
!python -m pytest tests -q -rs

## 6. Week 2 check: GPU vs CPU on chr21 through the CLI
Both TSV files must be byte-for-byte identical.

In [ ]:
G = "GAGTCCGAGCAGAAGAAGAA"
for mm in (4, 6):
    !python -m geneweaver scan --fasta data/chr21.fa --guide {G} --max-mm {mm} --backend gpu --chunk-size 5000000 --out results/chr21_gpu_mm{mm}.tsv --quiet
    !python -m geneweaver scan --fasta data/chr21.fa --guide {G} --max-mm {mm} --backend numba-cpu --chunk-size 5000000 --out results/chr21_cpu_mm{mm}.tsv --quiet
    !cmp results/chr21_gpu_mm{mm}.tsv results/chr21_cpu_mm{mm}.tsv && echo "max_mm={mm}: GPU == CPU ✅"

In [ ]:
!head -5 results/chr21_gpu_mm6.tsv | column -t